In [1]:
!nvidia-smi

import torch
print("CUDA available:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "No GPU")

Sun Jun 21 05:36:57 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   39C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
from google.colab import files
uploaded = files.upload()


Saving DSS_Blinkit_Project.zip to DSS_Blinkit_Project.zip


In [12]:
!unzip -q DSS_Blinkit_Project.zip -d /content/
%cd /content/DSS_Blinkit_Project
!ls

unzip:  cannot find or open DSS_Blinkit_Project.zip, DSS_Blinkit_Project.zip.zip or DSS_Blinkit_Project.zip.ZIP.
/content/DSS_Blinkit_Project
02_regression_forecasting_models.py
03_regression_forecasting_improved.py
04_inventory_decision_support.py
04_product_portfolio_decision_support.py
05_product_future_performance_classification.py
06_product_future_performance_classification_final.py
07_product_future_performance_classification_tabpfn.py
08_colab_gpu_benchmark.py
08_product_future_performance_extended_benchmark.py
blinkit_data2
blinkit_marketing_performance_clean.csv
blinkit_orders_clean.csv
catboost_info
figures
figures_benchmark08
figures_benchmark08_colab
member3_forecast_prediction_ready_fixed.ipynb
outputs
outputs_benchmark08
outputs_benchmark08_colab
README.md


In [4]:
%cd /content/DSS_Blinkit_Project
!ls

/content/DSS_Blinkit_Project
02_regression_forecasting_models.py
03_regression_forecasting_improved.py
04_inventory_decision_support.py
04_product_portfolio_decision_support.py
05_product_future_performance_classification.py
06_product_future_performance_classification_final.py
07_product_future_performance_classification_tabpfn.py
08_product_future_performance_extended_benchmark.py
blinkit_data2
blinkit_marketing_performance_clean.csv
blinkit_orders_clean.csv
catboost_info
figures
figures_benchmark08
member3_forecast_prediction_ready_fixed.ipynb
outputs
outputs_benchmark08
README.md


In [5]:
!cp 08_product_future_performance_extended_benchmark.py 08_colab_gpu_benchmark.py

In [6]:
from pathlib import Path
import re

path = Path("08_colab_gpu_benchmark.py")
text = path.read_text(encoding="utf-8")

# Đổi thư mục output riêng để không lẫn với kết quả local
text = text.replace('OUTPUT_DIR = "outputs_benchmark08"', 'OUTPUT_DIR = "outputs_benchmark08_colab"')
text = text.replace('FIGURE_DIR = "figures_benchmark08"', 'FIGURE_DIR = "figures_benchmark08_colab"')

marker = "    model_specs = get_classification_models(preprocessor)"

new_block = '''    model_specs = get_classification_models(preprocessor)

    # Colab GPU run:
    # - Skip TabPFN because it requires license/login/model-weight authentication.
    # - Do not skip TabM here because Colab GPU is used to test TabM.
    SKIP_MODELS = [
        "TabPFN Classifier 2025",
    ]

    for skipped_model in SKIP_MODELS:
        if skipped_model in model_specs:
            print(f"Skip model on Colab: {skipped_model}")
            model_specs.pop(skipped_model)
'''

# Nếu file đã có block SKIP_MODELS ngay sau model_specs thì thay block đó.
pattern = r'''    model_specs = get_classification_models\(preprocessor\)

    #.*?\n(?:    #.*?\n)*    SKIP_MODELS = \[
[\s\S]*?    for skipped_model in SKIP_MODELS:
        if skipped_model in model_specs:
            print\(.*?\)
            model_specs\.pop\(skipped_model\)
'''

if re.search(pattern, text):
    text = re.sub(pattern, new_block, text, count=1)
elif marker in text:
    text = text.replace(marker, new_block, 1)
else:
    raise ValueError("Không tìm thấy dòng model_specs = get_classification_models(preprocessor)")

path.write_text(text, encoding="utf-8")

print("Đã tạo và sửa file:", path)

Đã tạo và sửa file: 08_colab_gpu_benchmark.py


In [7]:
!nvidia-smi

import torch
print("CUDA available:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "No GPU")

Sun Jun 21 06:06:46 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   39C    P8              9W /   70W |       3MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [8]:
!pip install -q lightgbm xgboost catboost tabm

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.1/97.1 MB 8.8 MB/s eta 0:00:00


In [9]:
!python 08_colab_gpu_benchmark.py

Đã đọc dữ liệu:
Orders clean: (5000, 11) - ./blinkit_orders_clean.csv
Order items:  (5000, 4) - blinkit_data2/blinkit_order_items.csv
Products:     (268, 10) - blinkit_data2/blinkit_products.csv

Kiểm tra tháng dữ liệu:
Ngày đầu: 2023-03-16 08:10:44
Ngày cuối: 2024-11-04 20:29:15
Loại tháng không đầy đủ: 2023-03, 2024-11
Dải tháng dùng cho mô hình: 2023-04 đến 2024-10

Dataset supervised:
Total rows: 2948
Train rows: 2144
Test rows: 804
Train folds: [np.int64(6), np.int64(7), np.int64(8), np.int64(9), np.int64(10), np.int64(11), np.int64(12), np.int64(13)]
Test folds: [np.int64(14), np.int64(15), np.int64(16)]
Train positive rate: 0.2500
Test positive rate: 0.2500
Skip slow/unstable model: TabM 2025 Experimental
Training classifier with tuning: Logistic Regression Balanced
Training classifier with tuning: Linear SVM Balanced
Training classifier with tuning: Decision Tree Balanced
Training classifier with tuning: Random Forest Balanced
Training classifier with tuning: Extra Trees Balanc

In [10]:
import pandas as pd

pd.set_option("display.max_columns", None)

df = pd.read_csv("outputs_benchmark08_colab/final_classification_results.csv")
print(df.round(4).to_string(index=False))

                       model  threshold  Accuracy  Balanced_Accuracy  Precision  Recall  Specificity     F1  ROC_AUC  PR_AUC  Brier_Score  Top10_K  Precision_at_Top10pct  Recall_at_Top10pct  Lift_at_Top10pct  Top25_K  Precision_at_Top25pct  Recall_at_Top25pct  Lift_at_Top25pct  TN  FP  FN  TP  best_cv_f1  validation_threshold_f1  validation_threshold_precision  validation_threshold_recall                                                                                                                                 best_params  train_time_seconds  predict_time_seconds  predict_time_ms
      Random Forest Balanced       0.54    0.8134             0.7794     0.6085  0.7114       0.8474 0.6560   0.8742  0.6055       0.1272       81                 0.6420              0.2587            2.5679      201                 0.6119              0.6119            2.4478 511  92  58 143      0.6709                   0.6892                          0.6296                       0.7612                  

In [11]:
import pandas as pd

skip_path = "outputs_benchmark08_colab/benchmark08_skipped_models.csv"

try:
    skip = pd.read_csv(skip_path)
    print(skip.to_string(index=False))
except FileNotFoundError:
    print("Không có file skipped models.")

 model                                                                                                                   reason
TabPFN Tắt mặc định để tránh yêu cầu đăng nhập/license và lỗi socket Windows; có thể bật ENABLE_TABPFN=True nếu muốn thử riêng.
